# 36 Reference models on dev and mini-dev (A100)

Published models, run in **our** harness so that their numbers are directly comparable with ours (same questions, same schema information, same scoring, and our cost measure, which their papers do not report). Each model gets the prompt it was published with, greedy answers:

| Run | Model | Prompt | Published BIRD dev |
|---|---|---|---|
| `ref_omnisql7b` | OmniSQL-7B: Qwen2.5-Coder-7B-Instruct fully fine-tuned on 2.5M synthetic examples plus BIRD and Spider train | OmniSQL's, reasons before the SQL | 63.9 |
| `ref_arctic7b` | Arctic-Text2SQL-R1-7B: RL with an execution reward, starting from OmniSQL | Arctic's, reasons in `<think>` | 68.9 |
| `ref_qwen32b` | Qwen2.5-Coder-32B-Instruct, untuned (4-bit AWQ) | ours (zero-shot) | 64.5 (full precision, OmniSQL's prompt) |
| `ref_qwen7b_omnisql` | our base model, untuned | OmniSQL's | 50.9 (ours with our prompt: 53.9) |

Only predictions are made here; `37_reference_evaluate` scores them on a CPU runtime. Nothing is trained and nothing here is used to choose any of our models.

**Runtime:** A100. About 1 to 1.5 hours in total (most of it the reasoning models, which write several hundred tokens per answer), plus about 15 GB of download per 7B model and 19 GB for the 32B model. **Resumable:** after a disconnect, run the setup and config cells, then the cell that was running (answers are saved in chunks of 256).

**After every code update, reopen this notebook from the GitHub link.**

In [ ]:
# ---- effsql setup cell: identical in every notebook (source: notebooks/setup_cell.py) ----
REPO_URL = "https://github.com/selimsidan/correct-is-not-enough"
REPO_DIR = "/content/correct-is-not-enough"
REF = "main"  # branch, tag or commit hash to run
GPU = True  # set True in GPU notebooks: installs the [gen] extra (vLLM)
DRIVE_ROOT = "/content/drive/MyDrive/586"
LOCAL_ROOT = "/content/bird_local"

import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")


def sh(*cmd, check=True):
    r = subprocess.run(cmd, text=True, capture_output=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"{' '.join(cmd)} failed:\n{r.stderr}")
    return r.stdout.strip()


if not os.path.isdir(REPO_DIR):
    sh("git", "clone", "--quiet", REPO_URL, REPO_DIR)
sh("git", "-C", REPO_DIR, "fetch", "--quiet", "--all", "--tags")
sh("git", "-C", REPO_DIR, "checkout", "--quiet", REF)
if sh("git", "-C", REPO_DIR, "rev-parse", "--verify", "--quiet", f"origin/{REF}", check=False):
    sh("git", "-C", REPO_DIR, "merge", "--ff-only", "--quiet", f"origin/{REF}")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR + ("[gen]" if GPU else "")], check=True
)
os.environ["EFFSQL_DRIVE_ROOT"] = DRIVE_ROOT
os.environ["EFFSQL_LOCAL_ROOT"] = LOCAL_ROOT
print("effsql commit:", sh("git", "-C", REPO_DIR, "rev-parse", "HEAD"))

In [ ]:
# Run order: the two published 7B text-to-SQL models first, then the 32B model, then our base model with
# OmniSQL's prompt. Keys of configs/reference.yaml; each writes runs/ref_<key>.
MODELS = ["omnisql7b", "arctic7b", "qwen32b", "qwen7b_omnisql"]
R = f"{DRIVE_ROOT}/effsql/runs"
import os
assert os.path.isdir(f"{DRIVE_ROOT}/effsql/cache/schemas/dev"), "run 00_setup_and_stage first"
!nvidia-smi --query-gpu=name,memory.total --format=csv

### 1. OmniSQL-7B

In [ ]:
!python -m effsql.gen.reference --model omnisql7b

### 2. Arctic-Text2SQL-R1-7B (the longest: answers reason first)

In [ ]:
!python -m effsql.gen.reference --model arctic7b

### 3. Qwen2.5-Coder-32B-Instruct (AWQ), our zero-shot prompt

In [ ]:
!python -m effsql.gen.reference --model qwen32b

### 4. Our base model (Qwen2.5-Coder-7B-Instruct) with OmniSQL's prompt

In [ ]:
!python -m effsql.gen.reference --model qwen7b_omnisql

### 5. Check: answer length, cut-off answers and one example answer per model

In [ ]:
# Run order: the two published 7B text-to-SQL models first, then the 32B model, then our base model with
# OmniSQL's prompt. Keys of configs/reference.yaml; each writes runs/ref_<key>.
MODELS = ["omnisql7b", "arctic7b", "qwen32b", "qwen7b_omnisql"]
R = f"{DRIVE_ROOT}/effsql/runs"
import os
import json
for m in MODELS:
    for split in ("dev", "minidev"):
        f = f"{R}/ref_{m}/predict_stats_{split}.json"
        if os.path.exists(f):
            s = json.load(open(f))
            print(f"{m:<16}{split:<9}{s['n_questions']:>6} questions{s['mean_answer_tokens']:>8.0f} "
                  f"answer tokens on average{s['n_cut_off']:>5} cut off")
for m in MODELS:
    f = f"{R}/ref_{m}/predict_raw_dev.jsonl"
    if os.path.exists(f):
        r = json.loads(open(f).readline())
        print(f"\n===== {m} ({r['qid']}) =====\n" + r["completions"][0]["text"][-1500:])

### Done: disconnect now, then run `37_reference_evaluate` on a CPU runtime